# 🔁 Notebook 1: Retry Strategies

Networks flap. Servers restart. Sometimes calling again is all it takes.
But **how** and **when** you retry matters. Four common strategies:

1. **Immediate retry** — try right away.
2. **Fixed delay** — wait N seconds, try again.
3. **Exponential backoff** — wait 1, 2, 4, 8, ... seconds.
4. **Exponential + jitter** — backoff with a random ± component.

Only retry **idempotent** operations (see `04-patterns/idempotency`).

## 🛠️ Setup

```bash
cd 05-microservices/retry
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
import time, random

def flaky(p_fail=0.6):
    if random.random() < p_fail:
        raise RuntimeError('transient boom')
    return 'ok'

def retry(fn, strategy, max_attempts=6, **kw):
    for attempt in range(1, max_attempts+1):
        try:
            return fn(**kw), attempt
        except Exception as e:
            delay = strategy(attempt)
            print(f'  attempt {attempt} failed -> sleep {delay:.2f}s')
            time.sleep(delay)
    raise RuntimeError('gave up')


## Four strategies side-by-side

In [ ]:
strategies = {
    'immediate':   lambda a: 0,
    'fixed':       lambda a: 0.2,
    'exponential': lambda a: 0.1 * (2 ** (a-1)),
    'exp+jitter':  lambda a: 0.1 * (2 ** (a-1)) * random.uniform(0.5, 1.5),
}

random.seed(42)
for name, strat in strategies.items():
    print(f'--- {name} ---')
    try:
        result, n = retry(flaky, strat, max_attempts=5, p_fail=0.7)
        print(f'  succeeded on attempt {n}')
    except Exception as e:
        print(f'  {e}')


Notebook 2 shows why **jitter** matters: the thundering herd.